# QA pipeline eval

Bandingkan **skill QA** vs **Claude default tanpa skill**: hasil test dan token.

Tiap case adalah folder di `dataset/`: `conversation.jsonl` (percakapan membangun app dari nol, format transcript asli, termasuk tool call), `app/` (kode hasilnya), `bugs/*.patch` (bug regresi), `case.json` (golden). Dua arm, percakapan dan app ber-bug yang sama, cwd = salinan app:
- **skill**: transcript palsu → `qa.sh start` → `/qa-end` → "setuju" → `/qa-run`
- **baseline**: seluruh percakapan ditempel ke Claude biasa (skill/agent QA dimatikan) → "tolong test API"

Test list kedua arm dijalankan ulang oleh `qa.sh` ke app yang di-build ulang, lalu diskor dengan cara yang sama.

- `RUN_NEW = False`: buka run terakhir, **tanpa token**.
- `RUN_NEW = True`: jalankan `claude -p`, **pakai token**. ±$1.9 skill + ±$0.6 baseline per case di Opus (run lama, context pendek); per case dua arm ≈ **$2–3**. Mulai dari `CASES` kecil atau `ARMS = ["baseline"]`.
- Lihat `about` di tiap `dataset/*/case.json`. Validasi tanpa token: `python3 eval/run.py --check-dataset`.
- Logika ada di `run.py`. Notebook ini cuma antarmuka; CLI `python3 eval/run.py` tetap jalan.

In [1]:
RUN_NEW = False   # True = jalankan claude -p (pakai token). False = buka run terakhir
ARMS = ["skill", "baseline"]   # bisa salah satu saja untuk hemat
CASES = []        # kosong = semua folder di dataset/; contoh: ["01-user-service"]
MODEL = None      # None = default Claude Code-mu; contoh: "sonnet"
BUDGET = 2.0      # USD maks per panggilan claude
PORT = 18080      # port dummy API

In [2]:
import importlib
import sys
from pathlib import Path
from types import SimpleNamespace
from IPython.display import Markdown, display

E = Path.cwd() if (Path.cwd() / "run.py").exists() else Path.cwd() / "eval"
sys.path.insert(0, str(E))
import run as ev
importlib.reload(ev)  # ambil perubahan run.py tanpa restart kernel

print("install:", "OK" if not ev.stale_install() else f"BEDA {ev.stale_install()} -> jalankan bash install.sh")

install: OK


## 1. Jalankan atau buka run terakhir

In [3]:
a = SimpleNamespace(model=MODEL, budget=BUDGET, port=PORT, arms=ARMS)
if RUN_NEW:
    cases = [E / "dataset" / c for c in CASES] or ev.all_cases()
    OUT, scores = ev.run_cases(cases, a)
else:
    OUT = max(d for d in (E / "results").iterdir() if d.is_dir())  # nama folder = timestamp
    scores = [ev.load(d / "score.json") for d in sorted(OUT.iterdir()) if (d / "score.json").exists()]
print(OUT)

/Users/akyunnn/Documents/qa project/eval/results/20261028-104216


## 2. Ringkasan

In [4]:
display(Markdown(ev.summary(scores, OUT)))

# QA eval

## Skill: kualitas ekstraksi + queue

| Case | Recall | Type | Leak | Extra | Gate | Coverage | Untraced | Off-scope | Bugs caught | False pos | Inferred fail | Blocked | Tokens in / cache read / out | Cost |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 01-user-service | 13/13 | 13/13 | 0 | 6 | ok | 11/12 | 0 | 0 | 3/3 | 0 | 0 | 0 | 218608 / 1059285 / 42334 | $2.57 |

## Skill vs Claude default

Bug dan wrong fail dinilai dari hasil live, yaitu yang dilihat user (hasil /qa-run untuk skill, status yang dilaporkan untuk baseline). Tidak reproducible = test yang hasilnya berbeda saat daftar test-nya dijalankan ulang oleh qa.sh ke app yang di-build ulang.

| Case | Arm | Tests | Coverage | Expected = golden | Bugs caught | Wrong fails | Leak | Tidak reproducible | Tokens out | Cost | Waktu |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 01-user-service | skill | 26 | 11/12 | 12/12 | 3/3 | 0 | 0 | 0 | 42334 | $2.57 | 154s |
| 01-user-service | baseline | 39 | 9/12 | 11/12 | 3/3 | 0 | 0 | 6 | 17247 | $0.99 | 141s |

- **skill**: bugs 3/3 · wrong fails 0 · cost $2.57 · out 42334 · waktu 154s
- **baseline**: bugs 3/3 · wrong fails 0 · cost $0.99 · out 17247 · waktu 141s

### 01-user-service
- skill missed requirement: [] · missed bug: []
- skill leaks: [] · extra (cek manual, bisa halusinasi): ['G-1', 'G-2', 'C-1', 'C-2', 'C-5', 'C-6']
- skill false pos: [] · inferred fail: [] · blocked: []
- skill errors: [] · permission denials: 0
- skill expected beda/tidak ada vs golden: []
- baseline expected beda/tidak ada vs golden: ['E12']
- baseline wrong fails: [] · leaks: [] · tidak reproducible: ['B-6', 'B-26', 'B-32', 'B-33', 'B-34', 'B-35'] · permission denials: 4

## Token per bagian

Cost per panggilan = estimasi: cost asli tiap turn per model dibagi ke panggilannya sesuai bobot harga (input 1, cache write 1.25/2, cache read 0.1, output 5). Cost context = membaca system prompt, percakapan, hasil tool (naik tiap panggilan karena context makin panjang); cost output = yang ditulis model, termasuk thinking. Panggilan pertama tiap sesi memuat system prompt + CLAUDE.md + daftar skill (cache write besar). Semua baris ada di `steps.csv`.

### Per turn

| Arm / turn | Calls | Input | Cache write | Cache read | Output | Cost | Cost context | Cost output | % arm |
|---|---|---|---|---|---|---|---|---|---|
| baseline · baseline | 14 | 6933 | 62787 | 664216 | 17247 | $0.99 | $0.68 | $0.30 | 100% |
| skill · 1 /qa-end | 6 | 12 | 62631 | 142006 | 5235 | $0.61 | $0.51 | $0.10 | 24% |
| skill · 2 setuju -> queue | 3 | 8 | 20232 | 115401 | 11137 | $0.89 | $0.40 | $0.49 | 35% |
| skill · 3 /qa-run | 7 | 16 | 17480 | 419253 | 4649 | $1.07 | $0.91 | $0.17 | 42% |

### Per fase

| Arm / fase | Calls | Input | Cache write | Cache read | Output | Cost | Cost context | Cost output | % arm |
|---|---|---|---|---|---|---|---|---|---|
| baseline · 2 rancang test | 2 | 4 | 2545 | 115405 | 436 | $0.07 | $0.06 | $0.01 | 7% |
| baseline · 3 testing | 10 | 20 | 59732 | 486534 | 12366 | $0.81 | $0.59 | $0.22 | 82% |
| baseline · 4 report | 1 | 2 | 510 | 62277 | 4434 | $0.10 | $0.03 | $0.08 | 11% |
| baseline · lain (internal) | 1 | 6907 | 0 | 0 | 11 | $0.01 | $0.01 | $0.00 | 1% |
| skill · 1 define goal | 5 | 10 | 57304 | 93797 | 3117 | $0.51 | $0.45 | $0.06 | 20% |
| skill · 2 rancang test | 3 | 6 | 16464 | 158014 | 10427 | $0.94 | $0.43 | $0.50 | 36% |
| skill · 3 testing | 4 | 8 | 5707 | 270722 | 583 | $0.63 | $0.58 | $0.04 | 24% |
| skill · 4 report | 2 | 4 | 2678 | 142935 | 1238 | $0.39 | $0.30 | $0.09 | 15% |
| skill · lain (internal) | 2 | 8 | 18190 | 11192 | 5656 | $0.10 | $0.05 | $0.06 | 4% |

### Per subject (endpoint)

Biaya fase rancang test + testing dibagi ke endpoint sesuai jumlah test-nya (estimasi: `run-all` menjalankan semua test dalam satu panggilan, jadi token per test tidak tercatat terpisah).

| Arm / subject | Tests | Cost | % arm |
|---|---|---|---|
| baseline · POST /users | 14 | $0.31 | 32% |
| baseline · PATCH /users/{id} | 12 | $0.27 | 27% |
| baseline · GET /users/{id} | 7 | $0.16 | 16% |
| baseline · DELETE /users/{id} | 3 | $0.07 | 7% |
| baseline · GET /users/abc | 1 | $0.02 | 2% |
| baseline · PATCH /users/abc | 1 | $0.02 | 2% |
| baseline · DELETE /users/abc | 1 | $0.02 | 2% |
| skill · POST /users | 8 | $0.48 | 19% |
| skill · PATCH /users/{id} | 8 | $0.48 | 19% |
| skill · GET /users/{id} | 4 | $0.24 | 9% |
| skill · DELETE /users/{id} | 3 | $0.18 | 7% |
| skill · GET /users/abc | 1 | $0.06 | 2% |
| skill · PATCH /users/abc | 1 | $0.06 | 2% |
| skill · DELETE /users/abc | 1 | $0.06 | 2% |

### Per bagian (jenis panggilan)

| Arm / bagian | Calls | Input | Cache write | Cache read | Output | Cost | Cost context | Cost output | % arm |
|---|---|---|---|---|---|---|---|---|---|
| baseline · Bash curl | 7 | 14 | 51992 | 339493 | 6107 | $0.59 | $0.49 | $0.11 | 60% |
| baseline · jawaban teks | 3 | 6 | 3055 | 177682 | 4870 | $0.17 | $0.08 | $0.09 | 17% |
| baseline · Write qa.py | 1 | 2 | 2217 | 45922 | 4472 | $0.11 | $0.03 | $0.08 | 11% |
| baseline · Write qa.sh | 1 | 2 | 903 | 52980 | 1613 | $0.05 | $0.03 | $0.03 | 5% |
| baseline · Bash python3 | 1 | 2 | 4620 | 48139 | 174 | $0.05 | $0.05 | $0.00 | 5% |
| baseline · internal (tidak ada di transcript) | 1 | 6907 | 0 | 0 | 11 | $0.01 | $0.01 | $0.00 | 1% |
| skill · jawaban teks | 3 | 6 | 14234 | 177033 | 4620 | $0.63 | $0.44 | $0.20 | 25% |
| skill · Write queue.json | 1 | 2 | 2733 | 53536 | 6642 | $0.49 | $0.12 | $0.37 | 19% |
| skill · Bash qa.sh end | 1 | 2 | 43110 | 0 | 102 | $0.33 | $0.33 | $0.00 | 13% |
| skill · Bash qa.sh report | 1 | 2 | 2175 | 70380 | 403 | $0.20 | $0.17 | $0.03 | 8% |
| skill · Bash qa.sh approve | 1 | 2 | 2371 | 64673 | 95 | $0.18 | $0.17 | $0.01 | 7% |
| skill · Skill qa-run | 1 | 2 | 1936 | 67044 | 51 | $0.16 | $0.16 | $0.00 | 6% |
| skill · Bash jq | 1 | 2 | 1045 | 68980 | 274 | $0.16 | $0.14 | $0.02 | 6% |
| skill · Bash qa.sh run-all | 1 | 2 | 355 | 70025 | 163 | $0.13 | $0.12 | $0.01 | 5% |
| skill · internal (tidak ada di transcript) | 2 | 8 | 18190 | 11192 | 5656 | $0.10 | $0.05 | $0.06 | 4% |
| skill · Write requirements.md | 1 | 2 | 3118 | 45091 | 2733 | $0.09 | $0.04 | $0.05 | 4% |
| skill · Agent qa-goal-extractor | 1 | 2 | 1981 | 43110 | 184 | $0.04 | $0.03 | $0.00 | 1% |
| skill · subagent Read segment.txt | 1 | 2 | 5596 | 0 | 95 | $0.03 | $0.03 | $0.00 | 1% |
| skill · subagent jawaban teks | 1 | 2 | 3499 | 5596 | 3 | $0.02 | $0.02 | $0.00 | 1% |

### 8 panggilan termahal

| Case | Arm / turn | # | Label | Output | Cache read | Cost |
|---|---|---|---|---|---|---|
| 01-user-service | skill · 2 setuju -> queue | 7 | Write queue.json (+thinking) | 6642 | 53536 | $0.49 |
| 01-user-service | skill · 2 setuju -> queue | 8 | jawaban teks (+thinking) | 1667 | 56269 | $0.34 |
| 01-user-service | skill · 1 /qa-end | 1 | Bash qa.sh end | 102 | 0 | $0.33 |
| 01-user-service | baseline · baseline | 1 | Bash curl (+thinking) | 184 | 0 | $0.33 |
| 01-user-service | skill · 3 /qa-run | 13 | Bash qa.sh report (+thinking) | 403 | 70380 | $0.20 |
| 01-user-service | skill · 3 /qa-run | 14 | jawaban teks (+thinking) | 835 | 72555 | $0.19 |
| 01-user-service | skill · 3 /qa-run | 9 | Bash qa.sh approve | 95 | 64673 | $0.18 |
| 01-user-service | skill · 3 /qa-run | 10 | Skill qa-run | 51 | 67044 | $0.16 |


Cara baca:
- **Recall**: requirement golden yang ditemukan extractor. **Type**: `stated`/`inferred` benar.
- **Leak**: isi terlarang (pesan sebelum `/qa-start`, angka karangan) masuk requirement. Harus 0.
- **Extra**: baris G/C di luar golden. Cek manual: konteks sah atau halusinasi.
- **Gate**: skill menunggu konfirmasi sebelum bikin queue, dan queue belum `approved`.
- **Coverage**: requirement in-scope yang punya ≥1 test. **Untraced**: test tanpa ID requirement valid. **Off-scope**: test jenis di luar argumen `/qa-end`.
- **Bugs caught**: bug yang ditanam di `dummy_api.py` yang tertangkap test fail. Ini ukuran efektif utama.
- **False pos**: test `stated` fail padahal API benar (skill salah). **Inferred fail**: asumsi skill (`inferred`) yang salah. Wajar, tapi harus dikonfirmasi user.

Tabel **Skill vs Claude default**:
- **Coverage**: requirement golden yang punya test, dicocokkan dari teks test (title, path, expected). Sama untuk dua arm.
- **Expected = golden**: requirement yang test-nya mengirim request sama dan mengharapkan status sama dengan expected output golden (`output` di `case.json`, id di path dianggap sama). Yang tidak cocok tercatat di detail per case.
- **Wrong fails**: test fail padahal bagian API itu benar (ekspektasi salah).
- **Claim≠actual**: baseline melaporkan pass/fail berbeda dari hasil jalan ulang (Claude salah lapor).
- **Leak**: test menyentuh hal terlarang, misal endpoint `/legacy` dari pesan sebelum `/qa-start`.

## 3. Token per bagian

Tabel per turn, fase, subject, dan bagian ada di ringkasan (bagian 2). Di sini: bar per fase, bar per subject (endpoint), dan semua panggilan satu case. Data mentah: `steps.csv` di folder run.

In [5]:
steps = ev.all_steps(scores)
for arm in ("skill", "baseline"):
    rows = [r for r in steps if r["arm"] == arm]
    if not rows:
        continue
    parts = {}
    for r in rows:
        g = parts.setdefault(r["phase"], [0.0, 0.0, 0])
        g[0] += r["cost"]; g[1] += r["cost_out"]; g[2] += r["out"]
    total = sum(g[0] for g in parts.values()) or 1
    print(f"{arm}: total ${total:.2f}   (█ = context, ▒ = output)")
    for part, (cost, cost_out, out) in sorted(parts.items(), key=lambda kv: ev.PHASES.index(kv[0])):
        ctx = round((cost - cost_out) / total * 40); o = round(cost_out / total * 40)
        print(f"  {part[:34]:<34} {'█' * ctx}{'▒' * o:<{40 - ctx}} ${cost:.2f} ({100 * cost / total:.0f}%)  out {out}")
    subs = {}
    for sc in scores:
        for x in (sc.get("subjects", []) if arm == "skill" else sc.get("baseline", {}).get("subjects", [])):
            g = subs.setdefault(x["subject"], [0, 0.0]); g[0] += x["tests"]; g[1] += x["cost"]
    if subs:
        print(f"  per subject (rancang + testing, dibagi per jumlah test):")
        mx = max(c for _, c in subs.values()) or 1
        for sub, (n, cost) in sorted(subs.items(), key=lambda kv: -kv[1][1]):
            print(f"    {sub[:32]:<32} {'█' * round(cost / mx * 25):<25} ${cost:.2f}  {n} test")
    print()

skill: total $2.57   (█ = context, ▒ = output)
  1 define goal                      ███████▒                                 $0.51 (20%)  out 3117
  2 rancang test                     ███████▒▒▒▒▒▒▒▒                          $0.94 (36%)  out 10427
  3 testing                          █████████▒                               $0.63 (24%)  out 583
  4 report                           █████▒                                   $0.39 (15%)  out 1238
  lain (internal)                    █▒                                       $0.10 (4%)  out 5656
  per subject (rancang + testing, dibagi per jumlah test):
    POST /users                      █████████████████████████ $0.48  8 test
    PATCH /users/{id}                █████████████████████████ $0.48  8 test
    GET /users/{id}                  ████████████              $0.24  4 test
    DELETE /users/{id}               █████████                 $0.18  3 test
    GET /users/abc                   ███                       $0.06  1 test
    PATCH 

In [6]:
CASE_STEPS = scores[0]["case"]   # ganti untuk lihat case lain
rows = ["| # | Arm / turn | Fase | Model | Label | In | Cache write | Cache read | Out | Cost |", "|---|---|---|---|---|---|---|---|---|---|"]
for r in [r for r in steps if r["case"] == CASE_STEPS]:
    rows.append(f"| {r['n']} | {r['arm']} · {r['turn']} | {r['phase']} | {r['model']} | {r['label']} | {r['in']} | {r['cache_write']} "
                f"| {r['cache_read']} | {r['out']} | ${r['cost']:.3f} |")
display(Markdown("\n".join(rows)))

| # | Arm / turn | Fase | Model | Label | In | Cache write | Cache read | Out | Cost |
|---|---|---|---|---|---|---|---|---|---|
| 1 | skill · 1 /qa-end | 1 define goal | claude-opus-5-5 | Bash qa.sh end | 2 | 43110 | 0 | 102 | $0.331 |
| 2 | skill · 1 /qa-end | 1 define goal | claude-opus-5-5 | Agent qa-goal-extractor | 2 | 1981 | 43110 | 184 | $0.035 |
| 3 | skill · 1 /qa-end | 1 define goal | claude-sonnet-5 | Read segment.txt | 2 | 5596 | 0 | 95 | $0.031 |
| 4 | skill · 1 /qa-end | 1 define goal | claude-sonnet-5 | jawaban teks (+thinking) | 2 | 3499 | 5596 | 3 | $0.021 |
| 5 | skill · 1 /qa-end | 1 define goal | claude-opus-5-5 | Write requirements.md + Skill qa-requirements (+thinking) | 2 | 3118 | 45091 | 2733 | $0.093 |
| 6 | skill · 1 /qa-end | 2 rancang test | claude-opus-5-5 | jawaban teks (+thinking) | 2 | 5327 | 48209 | 2118 | $0.099 |
| 7 | skill · 2 setuju -> queue | 2 rancang test | claude-opus-5-5 | Write queue.json (+thinking) | 2 | 2733 | 53536 | 6642 | $0.492 |
| 8 | skill · 2 setuju -> queue | 2 rancang test | claude-opus-5-5 | jawaban teks (+thinking) | 2 | 8404 | 56269 | 1667 | $0.344 |
| 9 | skill · 3 /qa-run | 3 testing | claude-opus-5-5 | Bash qa.sh approve | 2 | 2371 | 64673 | 95 | $0.178 |
| 10 | skill · 3 /qa-run | 3 testing | claude-opus-5-5 | Skill qa-run | 2 | 1936 | 67044 | 51 | $0.165 |
| 11 | skill · 3 /qa-run | 3 testing | claude-opus-5-5 | Bash jq (+thinking) | 2 | 1045 | 68980 | 274 | $0.157 |
| 12 | skill · 3 /qa-run | 3 testing | claude-opus-5-5 | Bash qa.sh run-all | 2 | 355 | 70025 | 163 | $0.130 |
| 13 | skill · 3 /qa-run | 4 report | claude-opus-5-5 | Bash qa.sh report (+thinking) | 2 | 2175 | 70380 | 403 | $0.204 |
| 14 | skill · 3 /qa-run | 4 report | claude-opus-5-5 | jawaban teks (+thinking) | 2 | 503 | 72555 | 835 | $0.189 |
| 15 | skill · 2 setuju -> queue | lain (internal) | claude-sonnet-5 | internal claude-sonnet-5 | 4 | 9095 | 5596 | 2828 | $0.052 |
| 16 | skill · 3 /qa-run | lain (internal) | claude-sonnet-5 | internal claude-sonnet-5 | 4 | 9095 | 5596 | 2828 | $0.052 |
| 1 | baseline · baseline | 3 testing | claude-opus-5-5 | Bash curl (+thinking) | 2 | 45922 | 0 | 184 | $0.327 |
| 2 | baseline · baseline | 3 testing | claude-opus-5-5 | Write qa.py (+thinking) | 2 | 2217 | 45922 | 4472 | $0.111 |
| 3 | baseline · baseline | 3 testing | claude-opus-5-5 | Bash python3 | 2 | 4620 | 48139 | 174 | $0.053 |
| 4 | baseline · baseline | 3 testing | claude-opus-5-5 | Bash curl (+thinking) | 2 | 221 | 52759 | 844 | $0.035 |
| 5 | baseline · baseline | 3 testing | claude-opus-5-5 | Write qa.sh (+thinking) | 2 | 903 | 52980 | 1613 | $0.053 |
| 6 | baseline · baseline | 2 rancang test | claude-opus-5-5 | jawaban teks + Bash bash | 2 | 1790 | 53883 | 211 | $0.035 |
| 7 | baseline · baseline | 3 testing | claude-opus-5-5 | Bash curl (+thinking) | 2 | 258 | 55673 | 934 | $0.038 |
| 8 | baseline · baseline | 3 testing | claude-opus-5-5 | Bash curl (+thinking) | 2 | 1002 | 55931 | 1280 | $0.049 |
| 9 | baseline · baseline | 3 testing | claude-opus-5-5 | Bash curl | 2 | 1410 | 56933 | 1104 | $0.049 |
| 10 | baseline · baseline | 3 testing | claude-opus-5-5 | Bash curl (+thinking) | 2 | 1511 | 58343 | 1244 | $0.053 |
| 11 | baseline · baseline | 3 testing | claude-opus-5-5 | Bash curl (+thinking) | 2 | 1668 | 59854 | 517 | $0.042 |
| 12 | baseline · baseline | 2 rancang test | claude-opus-5-5 | jawaban teks + Grep (+thinking) | 2 | 755 | 61522 | 225 | $0.031 |
| 13 | baseline · baseline | 4 report | claude-opus-5-5 | jawaban teks (+thinking) | 2 | 510 | 62277 | 4434 | $0.104 |
| 14 | baseline · baseline | lain (internal) | claude-haiku-4-5-20251001 | internal claude-haiku-4-5-20251001 | 6907 | 0 | 0 | 11 | $0.007 |

## 4. Detail satu case

In [7]:
CASE = scores[0]["case"]   # ganti untuk lihat case lain
d = OUT / CASE
req = d / "proj" / "requirements.md"
display(Markdown(req.read_text() if req.exists() else "_requirements.md tidak ada_"))

cell = lambda x: str(x if x is not None else "").replace("|", "\\|").replace("\n", " ")[:90]

def tests_table(title, qf):
    q = ev.load(qf) or {}
    rows = [f"**{title}**", "", "| ID | Req | Inferred | Request | Expected | Claimed | Status | Actual |", "|---|---|---|---|---|---|---|---|"]
    for t in q.get("tests", []):
        rows.append(f"| {t['id']} | {t.get('requirement_id')} | {'ya' if t.get('inferred') else ''} "
                    f"| {cell(t.get('method'))} {cell(t.get('path'))} | {cell(t.get('expect_status'))} {cell(t.get('expect_jq'))} "
                    f"| {cell(t.get('claimed'))} | {t.get('status')} | {cell(t.get('actual'))} |")
    display(Markdown("\n".join(rows) if q else f"**{title}**: _tidak ada test_"))

tests_table("Skill", d / "proj" / ".qa" / "queue.json")
tests_table("Baseline (Claude default)", d / "baseline" / "proj" / ".qa" / "queue.json")

# Requirements

## Goals
| ID | Requirement | Sumber | Tipe | Confidence |
|---|---|---|---|---|
| G-1 | Service users untuk aplikasi toko online, Go, net/http standard library tanpa dependency | USER: "Pakai Go, cukup net/http standard library" | stated | high |
| G-2 | 4 endpoint: GET /users/{id}, POST /users, PATCH /users/{id}, DELETE /users/{id} | USER: "Fokus 4 endpoint itu" | stated | high |

## Constraint
| ID | Requirement | Sumber | Tipe | Confidence |
|---|---|---|---|---|
| C-1 | Storage in-memory dulu, Postgres nanti setelah API stabil | USER: "Untuk lokal in-memory dulu, Postgres nanti" | stated | high |
| C-2 | Base URL http://127.0.0.1:18080 (port default 18080, bisa override via PORT) | USER: "Port lokal 18080, jadi base URL http://127.0.0.1:18080" | stated | high |
| C-3 | Semua error response format {error: pesan}, pesan bebas | USER: "Semua error di service ini formatnya {error: pesan}" | stated | high |
| C-4 | Seed data tetap: user id 1, email exists@test.co, name Seed | USER: "Seed satu user buat testing... Itu dipakai terus ya" | stated | high |
| C-5 | Tanpa autentikasi (akses internal lewat gateway) | USER: "Belum, service ini cuma diakses internal lewat gateway. Tanpa auth dulu" | stated | high |
| C-6 | Belum ada target response time, akan ada load test dari infra terpisah | USER: "Belum perlu target. Nanti ada load test dari infra" | stated | high |

## Keputusan yang sudah diambil
| ID | Requirement | Sumber | Tipe | Confidence |
|---|---|---|---|---|
| D-1 | GET /users/{id} sukses balas 200 JSON {id, email, name} | USER: "Balas 200 dengan JSON id, email, name" | stated | high |
| D-2 | GET /users/{id} user tidak ketemu balas 404 | USER: "404 dengan body {error: pesan}" | stated | high |
| D-3 | GET /users/{id} dengan id bukan angka (mis. /users/abc) balas 404, bukan 400 | USER: "tim mobile maunya 404 aja biar simpel" (mengalahkan usulan assistant 400) | stated | high |
| D-4 | POST /users body {email, name}, sukses balas 201 dengan user termasuk id | USER: "sukses 201 dengan user yang dibuat termasuk id" | stated | high |
| D-5 | POST /users: email wajib, format valid minimal ada @ dan titik di domain, tanpa cek MX; gagal balas 400 | USER: "Email wajib dan formatnya valid, minimal ada @ dan titik... Gak usah cek MX" | stated | high |
| D-6 | POST /users: name wajib, tidak boleh kosong/hanya spasi, gagal balas 400 | USER: "Name wajib. Kosong atau cuma spasi juga 400" | stated | high |
| D-7 | Email harus unik secara global (tanpa konsep tenant), duplikat balas 409 | USER: "Iya, email unik. Duplikat balas 409" / "Global. Tidak ada konsep tenant" | stated | high |
| D-8 | Keunikan email tidak peduli huruf besar/kecil (case-insensitive, pakai lowercase) — bug fix: EXISTS@test.co dulu lolos duplikat, sekarang harus kena 409 | USER: "jadi aturannya: unik email tidak peduli huruf besar kecil" | stated | high |
| D-9 | PATCH /users/{id} adalah partial update: field yang tidak dikirim tidak boleh berubah | USER: "Jadi update sebagian: field yang tidak dikirim tidak boleh berubah" (mengalahkan asumsi awal replace semua) | stated | high |
| D-10 | PATCH sukses balas 200 dengan user terbaru | USER: "Iya. User tidak ada 404..." (menjawab "Balas 200 dengan user terbaru?") | stated | high |
| D-11 | PATCH: user tidak ada balas 404 | USER: "User tidak ada 404" | stated | high |
| D-12 | PATCH: email baru format salah balas 400 | USER: "Email baru formatnya salah 400" | stated | high |
| D-13 | PATCH: email baru sudah dipakai user lain balas 409 | USER: "Email baru yang sudah dipakai user lain 409" | stated | high |
| D-14 | PATCH: name jika dikirim juga tidak boleh kosong | USER: "Name kalau dikirim juga tidak boleh kosong" | stated | high |
| D-15 | Bug fix: PATCH hanya field yang dikirim yang berubah; field tidak dikirim (nil) tidak boleh ikut ter-reset jadi kosong (regresi email ikut kosong saat PATCH name) | USER: "PATCH name doang kok email-nya jadi kosong?" | stated | high |
| D-16 | DELETE /users/{id} adalah hard delete (data benar-benar dihapus dari store, bukan soft delete/flag) | USER: "data user harus benar-benar dihapus. Ganti jadi hard delete" (mengalahkan keputusan awal soft delete) | stated | high |
| D-17 | DELETE sukses balas 204 tanpa body | USER: "DELETE balas 204 tanpa body" | stated | high |
| D-18 | Setelah DELETE, GET user tersebut harus 404 (bug fix: sebelumnya masih 200 karena map users tidak ikut terhapus) | USER: "Setelah dihapus, GET user itu harus 404" / "GET user itu masih 200. Kok masih ada?" | stated | high |
| D-19 | DELETE user yang tidak ada balas 404 | USER: "Hapus user yang tidak ada juga 404" | stated | high |
| D-20 | GET /users (list semua user dengan pagination) tidak dikerjakan dulu, belum ada kebutuhan | USER: "Jangan dulu, belum ada kebutuhan" | stated | high |
| D-21 | Tombol Daftar disable selama request jalan adalah tanggung jawab tim FE, bukan bagian dari backend API ini | USER: "Itu dikerjakan tim FE, catat aja" | stated | high |

## Belum jelas
| ID | Requirement | Sumber | Tipe | Confidence |
|---|---|---|---|---|
| U-1 | Apakah ada aturan tambahan validasi POST/PATCH selain email & name (misal panjang maksimum field, karakter terlarang)? | - | inferred | low |
| U-2 | Kapan migrasi ke PostgreSQL akan dilakukan dan apakah ada perubahan kontrak API saat migrasi? | USER: "Postgres nanti setelah API-nya stabil" | inferred | low |


**Skill**

| ID | Req | Inferred | Request | Expected | Claimed | Status | Actual |
|---|---|---|---|---|---|---|---|
| T-001 | D-1 |  | GET /users/1 | 200 .id == 1 and .email == "exists@test.co" and .name == "Seed" |  | pass | 200 {"id":1,"email":"exists@test.co","name":"Seed"}  |
| T-002 | D-2 |  | GET /users/999999 | 404 .error \| type == "string" |  | pass | 404 {"error":"user not found"}  |
| T-003 | D-3 |  | GET /users/abc | 404 .error \| type == "string" |  | pass | 404 {"error":"user not found"}  |
| T-004 | D-4 |  | POST /users | 201 .id != null and .email == "qa.user1@test.co" and .name == "QA User" |  | pass | 201 {"id":2,"email":"qa.user1@test.co","name":"QA User"}  |
| T-005 | D-5 |  | POST /users | 400 .error \| type == "string" |  | pass | 400 {"error":"invalid email"}  |
| T-006 | D-5 |  | POST /users | 400 .error \| type == "string" |  | pass | 400 {"error":"invalid email"}  |
| T-007 | D-6 |  | POST /users | 400 .error \| type == "string" |  | pass | 400 {"error":"name required"}  |
| T-008 | D-6 |  | POST /users | 400 .error \| type == "string" |  | pass | 400 {"error":"name required"}  |
| T-009 | D-7 |  | POST /users | 409 .error \| type == "string" |  | pass | 409 {"error":"email already used"}  |
| T-010 | D-8 |  | POST /users | 409 .error \| type == "string" |  | fail | 201 {"id":3,"email":"EXISTS@test.co","name":"Dup Upper"}  |
| T-011 | D-10 |  | PATCH /users/$U1 | 200 .name == "QA Renamed" |  | pass | 200 {"id":2,"email":"","name":"QA Renamed"}  |
| T-012 | D-15 |  | GET /users/$U1 | 200 .email == "qa.user1@test.co" and .name == "QA Renamed" |  | fail | 200 {"id":2,"email":"","name":"QA Renamed"}  |
| T-013 | D-9 |  | PATCH /users/$U1 | 200 .email == "qa.user1b@test.co" and .name == "QA Renamed" |  | fail | 200 {"id":2,"email":"qa.user1b@test.co","name":""}  |
| T-014 | D-11 |  | PATCH /users/999999 | 404 .error \| type == "string" |  | pass | 404 {"error":"user not found"}  |
| T-015 | D-11 | ya | PATCH /users/abc | 404 .error \| type == "string" |  | pass | 404 {"error":"user not found"}  |
| T-016 | D-12 |  | PATCH /users/$U1 | 400 .error \| type == "string" |  | pass | 400 {"error":"invalid email"}  |
| T-017 | D-13 |  | PATCH /users/$U1 | 409 .error \| type == "string" |  | pass | 409 {"error":"email already used"}  |
| T-018 | D-8 |  | PATCH /users/$U1 | 409 .error \| type == "string" |  | fail | 200 {"id":2,"email":"Exists@Test.CO","name":""}  |
| T-019 | D-14 |  | PATCH /users/$U1 | 400 .error \| type == "string" |  | pass | 400 {"error":"name required"}  |
| T-020 | D-14 | ya | PATCH /users/$U1 | 400 .error \| type == "string" |  | pass | 400 {"error":"name required"}  |
| T-021 | D-17 |  | DELETE /users/$U1 | 204  |  | pass | 204  |
| T-022 | D-18 |  | GET /users/$U1 | 404 .error \| type == "string" |  | fail | 200 {"id":2,"email":"Exists@Test.CO","name":""}  |
| T-023 | D-19 |  | DELETE /users/$U1 | 404 .error \| type == "string" |  | fail | 204  |
| T-024 | D-19 | ya | DELETE /users/abc | 404 .error \| type == "string" |  | pass | 404 {"error":"user not found"}  |
| T-025 | D-16 | ya | POST /users | 201 .id != null |  | pass | 201 {"id":4,"email":"qa.user1b@test.co","name":"QA Reuse"}  |
| T-026 | D-17 |  | DELETE /users/$U2 | 204  |  | pass | 204  |

**Baseline (Claude default)**

| ID | Req | Inferred | Request | Expected | Claimed | Status | Actual |
|---|---|---|---|---|---|---|---|
| B-1 | - |  | GET /users/1 | 200  | pass | pass | 200 {"id":1,"email":"exists@test.co","name":"Seed"}  |
| B-2 | - |  | GET /users/99999999 | 404  | pass | pass | 404 {"error":"user not found"}  |
| B-3 | - |  | GET /users/abc | 404  | pass | pass | 404 {"error":"user not found"}  |
| B-4 | - |  | GET /users/0 | 404  | pass | pass | 404 {"error":"user not found"}  |
| B-5 | - |  | POST /users | 201  | pass | pass | 201 {"id":2,"email":"alice.q1028@test.co","name":"Alice"}  |
| B-6 | - |  | GET /users/3 | 200  | pass | fail | 404 {"error":"user not found"}  |
| B-7 | - |  | POST /users | 400  | pass | pass | 400 {"error":"invalid email"}  |
| B-8 | - |  | POST /users | 400  | pass | pass | 400 {"error":"invalid email"}  |
| B-9 | - |  | POST /users | 400  | pass | pass | 400 {"error":"invalid email"}  |
| B-10 | - |  | POST /users | 400  | pass | pass | 400 {"error":"invalid email"}  |
| B-11 | - |  | POST /users | 400  | pass | pass | 400 {"error":"name required"}  |
| B-12 | - |  | POST /users | 400  | pass | pass | 400 {"error":"name required"}  |
| B-13 | - |  | POST /users | 400  | pass | pass | 400 {"error":"name required"}  |
| B-14 | - |  | POST /users | 400  | pass | pass | 400 {"error":"invalid json"}  |
| B-15 | - |  | POST /users | 409  | pass | pass | 409 {"error":"email already used"}  |
| B-16 | - |  | POST /users | 409  | fail | fail | 201 {"id":3,"email":"EXISTS@test.co","name":"Dup"}  |
| B-17 | - |  | POST /users | 409  | fail | fail | 201 {"id":4,"email":"ALICE.Q1028@TEST.CO","name":"Dup"}  |
| B-18 | - |  | PATCH /users/5 | 200  | fail | fail | 404 {"error":"user not found"}  |
| B-19 | - |  | GET /users/5 | 200  | fail | fail | 404 {"error":"user not found"}  |
| B-20 | - |  | PATCH /users/5 | 200  | fail | fail | 404 {"error":"user not found"}  |
| B-21 | - |  | PATCH /users/5 | 200  | fail | fail | 404 {"error":"user not found"}  |
| B-22 | - |  | PATCH /users/99999999 | 404  | pass | pass | 404 {"error":"user not found"}  |
| B-23 | - |  | PATCH /users/abc | 404  | pass | pass | 404 {"error":"user not found"}  |
| B-24 | - |  | PATCH /users/5 | 400  | pass | pass | 400 {"error":"invalid email"}  |
| B-25 | - |  | PATCH /users/5 | 400  | pass | pass | 400 {"error":"invalid email"}  |
| B-26 | - |  | PATCH /users/5 | 409  | pass | fail | 404 {"error":"user not found"}  |
| B-27 | - |  | PATCH /users/5 | 409  | fail | fail | 404 {"error":"user not found"}  |
| B-28 | - |  | PATCH /users/5 | 400  | pass | pass | 400 {"error":"name required"}  |
| B-29 | - |  | PATCH /users/5 | 400  | pass | pass | 400 {"error":"name required"}  |
| B-30 | - |  | PATCH /users/5 | 400  | pass | pass | 400 {"error":"invalid json"}  |
| B-31 | - |  | POST /users | 201  | pass | pass | 201 {"id":5,"email":"bob.q1028@test.co","name":"Reuse"}  |
| B-32 | - |  | DELETE /users/6 | 204  | pass | fail | 404 {"error":"user not found"}  |
| B-33 | - |  | GET /users/6 | 404  | fail | pass | 404 {"error":"user not found"}  |
| B-34 | - |  | PATCH /users/6 | 404  | fail | pass | 404 {"error":"user not found"}  |
| B-35 | - |  | DELETE /users/6 | 404  | fail | pass | 404 {"error":"user not found"}  |
| B-36 | - |  | DELETE /users/99999999 | 404  | pass | pass | 404 {"error":"user not found"}  |
| B-37 | - |  | DELETE /users/abc | 404  | pass | pass | 404 {"error":"user not found"}  |
| B-38 | - |  | POST /users | 201  | pass | pass | 201 {"id":6,"email":"carol.q1028@test.co","name":"Carol Again"}  |
| B-39 | - |  | GET /users/1 | 200  | pass | pass | 200 {"id":1,"email":"exists@test.co","name":"Seed"}  |

## 5. Skor ulang tanpa token

Setelah edit golden di `dataset/*.json` atau `run.py`, jalankan cell ini. API test dijalankan ulang ke dummy API, `claude` tidak dipanggil.

In [8]:
scores = ev.rescore(OUT, PORT)
display(Markdown(ev.summary(scores, OUT)))

{"case": "01-user-service", "skill_bugs": [3, 3], "baseline_bugs": [3, 3]}


# QA eval

## Skill: kualitas ekstraksi + queue

| Case | Recall | Type | Leak | Extra | Gate | Coverage | Untraced | Off-scope | Bugs caught | False pos | Inferred fail | Blocked | Tokens in / cache read / out | Cost |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 01-user-service | 13/13 | 13/13 | 0 | 6 | ok | 11/12 | 0 | 0 | 3/3 | 0 | 0 | 0 | 218608 / 1059285 / 42334 | $2.57 |

## Skill vs Claude default

Bug dan wrong fail dinilai dari hasil live, yaitu yang dilihat user (hasil /qa-run untuk skill, status yang dilaporkan untuk baseline). Tidak reproducible = test yang hasilnya berbeda saat daftar test-nya dijalankan ulang oleh qa.sh ke app yang di-build ulang.

| Case | Arm | Tests | Coverage | Expected = golden | Bugs caught | Wrong fails | Leak | Tidak reproducible | Tokens out | Cost | Waktu |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 01-user-service | skill | 26 | 11/12 | 12/12 | 3/3 | 0 | 0 | 0 | 42334 | $2.57 | 154s |
| 01-user-service | baseline | 39 | 9/12 | 11/12 | 3/3 | 0 | 0 | 6 | 17247 | $0.99 | 141s |

- **skill**: bugs 3/3 · wrong fails 0 · cost $2.57 · out 42334 · waktu 154s
- **baseline**: bugs 3/3 · wrong fails 0 · cost $0.99 · out 17247 · waktu 141s

### 01-user-service
- skill missed requirement: [] · missed bug: []
- skill leaks: [] · extra (cek manual, bisa halusinasi): ['G-1', 'G-2', 'C-1', 'C-2', 'C-5', 'C-6']
- skill false pos: [] · inferred fail: [] · blocked: []
- skill errors: [] · permission denials: 0
- skill expected beda/tidak ada vs golden: []
- baseline expected beda/tidak ada vs golden: ['E12']
- baseline wrong fails: [] · leaks: [] · tidak reproducible: ['B-6', 'B-26', 'B-32', 'B-33', 'B-34', 'B-35'] · permission denials: 4

## Token per bagian

Cost per panggilan = estimasi: cost asli tiap turn per model dibagi ke panggilannya sesuai bobot harga (input 1, cache write 1.25/2, cache read 0.1, output 5). Cost context = membaca system prompt, percakapan, hasil tool (naik tiap panggilan karena context makin panjang); cost output = yang ditulis model, termasuk thinking. Panggilan pertama tiap sesi memuat system prompt + CLAUDE.md + daftar skill (cache write besar). Semua baris ada di `steps.csv`.

### Per turn

| Arm / turn | Calls | Input | Cache write | Cache read | Output | Cost | Cost context | Cost output | % arm |
|---|---|---|---|---|---|---|---|---|---|
| baseline · baseline | 14 | 6933 | 62787 | 664216 | 17247 | $0.99 | $0.68 | $0.30 | 100% |
| skill · 1 /qa-end | 6 | 12 | 62631 | 142006 | 5235 | $0.61 | $0.51 | $0.10 | 24% |
| skill · 2 setuju -> queue | 3 | 8 | 20232 | 115401 | 11137 | $0.89 | $0.40 | $0.49 | 35% |
| skill · 3 /qa-run | 7 | 16 | 17480 | 419253 | 4649 | $1.07 | $0.91 | $0.17 | 42% |

### Per fase

| Arm / fase | Calls | Input | Cache write | Cache read | Output | Cost | Cost context | Cost output | % arm |
|---|---|---|---|---|---|---|---|---|---|
| baseline · 2 rancang test | 2 | 4 | 2545 | 115405 | 436 | $0.07 | $0.06 | $0.01 | 7% |
| baseline · 3 testing | 10 | 20 | 59732 | 486534 | 12366 | $0.81 | $0.59 | $0.22 | 82% |
| baseline · 4 report | 1 | 2 | 510 | 62277 | 4434 | $0.10 | $0.03 | $0.08 | 11% |
| baseline · lain (internal) | 1 | 6907 | 0 | 0 | 11 | $0.01 | $0.01 | $0.00 | 1% |
| skill · 1 define goal | 5 | 10 | 57304 | 93797 | 3117 | $0.51 | $0.45 | $0.06 | 20% |
| skill · 2 rancang test | 3 | 6 | 16464 | 158014 | 10427 | $0.94 | $0.43 | $0.50 | 36% |
| skill · 3 testing | 4 | 8 | 5707 | 270722 | 583 | $0.63 | $0.58 | $0.04 | 24% |
| skill · 4 report | 2 | 4 | 2678 | 142935 | 1238 | $0.39 | $0.30 | $0.09 | 15% |
| skill · lain (internal) | 2 | 8 | 18190 | 11192 | 5656 | $0.10 | $0.05 | $0.06 | 4% |

### Per subject (endpoint)

Biaya fase rancang test + testing dibagi ke endpoint sesuai jumlah test-nya (estimasi: `run-all` menjalankan semua test dalam satu panggilan, jadi token per test tidak tercatat terpisah).

| Arm / subject | Tests | Cost | % arm |
|---|---|---|---|
| baseline · POST /users | 14 | $0.31 | 32% |
| baseline · PATCH /users/{id} | 12 | $0.27 | 27% |
| baseline · GET /users/{id} | 7 | $0.16 | 16% |
| baseline · DELETE /users/{id} | 3 | $0.07 | 7% |
| baseline · GET /users/abc | 1 | $0.02 | 2% |
| baseline · PATCH /users/abc | 1 | $0.02 | 2% |
| baseline · DELETE /users/abc | 1 | $0.02 | 2% |
| skill · POST /users | 8 | $0.48 | 19% |
| skill · PATCH /users/{id} | 8 | $0.48 | 19% |
| skill · GET /users/{id} | 4 | $0.24 | 9% |
| skill · DELETE /users/{id} | 3 | $0.18 | 7% |
| skill · GET /users/abc | 1 | $0.06 | 2% |
| skill · PATCH /users/abc | 1 | $0.06 | 2% |
| skill · DELETE /users/abc | 1 | $0.06 | 2% |

### Per bagian (jenis panggilan)

| Arm / bagian | Calls | Input | Cache write | Cache read | Output | Cost | Cost context | Cost output | % arm |
|---|---|---|---|---|---|---|---|---|---|
| baseline · Bash curl | 7 | 14 | 51992 | 339493 | 6107 | $0.59 | $0.49 | $0.11 | 60% |
| baseline · jawaban teks | 3 | 6 | 3055 | 177682 | 4870 | $0.17 | $0.08 | $0.09 | 17% |
| baseline · Write qa.py | 1 | 2 | 2217 | 45922 | 4472 | $0.11 | $0.03 | $0.08 | 11% |
| baseline · Write qa.sh | 1 | 2 | 903 | 52980 | 1613 | $0.05 | $0.03 | $0.03 | 5% |
| baseline · Bash python3 | 1 | 2 | 4620 | 48139 | 174 | $0.05 | $0.05 | $0.00 | 5% |
| baseline · internal (tidak ada di transcript) | 1 | 6907 | 0 | 0 | 11 | $0.01 | $0.01 | $0.00 | 1% |
| skill · jawaban teks | 3 | 6 | 14234 | 177033 | 4620 | $0.63 | $0.44 | $0.20 | 25% |
| skill · Write queue.json | 1 | 2 | 2733 | 53536 | 6642 | $0.49 | $0.12 | $0.37 | 19% |
| skill · Bash qa.sh end | 1 | 2 | 43110 | 0 | 102 | $0.33 | $0.33 | $0.00 | 13% |
| skill · Bash qa.sh report | 1 | 2 | 2175 | 70380 | 403 | $0.20 | $0.17 | $0.03 | 8% |
| skill · Bash qa.sh approve | 1 | 2 | 2371 | 64673 | 95 | $0.18 | $0.17 | $0.01 | 7% |
| skill · Skill qa-run | 1 | 2 | 1936 | 67044 | 51 | $0.16 | $0.16 | $0.00 | 6% |
| skill · Bash jq | 1 | 2 | 1045 | 68980 | 274 | $0.16 | $0.14 | $0.02 | 6% |
| skill · Bash qa.sh run-all | 1 | 2 | 355 | 70025 | 163 | $0.13 | $0.12 | $0.01 | 5% |
| skill · internal (tidak ada di transcript) | 2 | 8 | 18190 | 11192 | 5656 | $0.10 | $0.05 | $0.06 | 4% |
| skill · Write requirements.md | 1 | 2 | 3118 | 45091 | 2733 | $0.09 | $0.04 | $0.05 | 4% |
| skill · Agent qa-goal-extractor | 1 | 2 | 1981 | 43110 | 184 | $0.04 | $0.03 | $0.00 | 1% |
| skill · subagent Read segment.txt | 1 | 2 | 5596 | 0 | 95 | $0.03 | $0.03 | $0.00 | 1% |
| skill · subagent jawaban teks | 1 | 2 | 3499 | 5596 | 3 | $0.02 | $0.02 | $0.00 | 1% |

### 8 panggilan termahal

| Case | Arm / turn | # | Label | Output | Cache read | Cost |
|---|---|---|---|---|---|---|
| 01-user-service | skill · 2 setuju -> queue | 7 | Write queue.json (+thinking) | 6642 | 53536 | $0.49 |
| 01-user-service | skill · 2 setuju -> queue | 8 | jawaban teks (+thinking) | 1667 | 56269 | $0.34 |
| 01-user-service | skill · 1 /qa-end | 1 | Bash qa.sh end | 102 | 0 | $0.33 |
| 01-user-service | baseline · baseline | 1 | Bash curl (+thinking) | 184 | 0 | $0.33 |
| 01-user-service | skill · 3 /qa-run | 13 | Bash qa.sh report (+thinking) | 403 | 70380 | $0.20 |
| 01-user-service | skill · 3 /qa-run | 14 | jawaban teks (+thinking) | 835 | 72555 | $0.19 |
| 01-user-service | skill · 3 /qa-run | 9 | Bash qa.sh approve | 95 | 64673 | $0.18 |
| 01-user-service | skill · 3 /qa-run | 10 | Skill qa-run | 51 | 67044 | $0.16 |
